# JazzCash Fraud Detection - Data Integration and Exploration

This notebook demonstrates the data integration process for combining IAR transactions, Mbar customer data, and fraud labels.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_integration.merge_datasets import DataIntegrator
from src.data_integration.data_cleaner import DataCleaner

# Configure display
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

%matplotlib inline

## 2. Load Datasets

In [ ]:
# Initialize integrator
integrator = DataIntegrator()

# Load datasets
iar_df, mbar_df, fraud_df = integrator.load_datasets(
    iar_path='../data/raw/iar_transactions.csv',
    mbar_path='../data/raw/mbar_customers.csv',
    fraud_path='../data/raw/fraud_labels.csv'
)

## 3. Explore Individual Datasets

In [ ]:
# IAR Transactions
print("IAR Transactions Schema:")
print(iar_df.info())
print("\nFirst few rows:")
display(iar_df.head())

In [ ]:
# Mbar Customers
print("Mbar Customers Schema:")
print(mbar_df.info())
print("\nFirst few rows:")
display(mbar_df.head())

In [ ]:
# Fraud Labels
print("Fraud Labels Schema:")
print(fraud_df.info())
print("\nFirst few rows:")
display(fraud_df.head())

## 4. Data Integration

In [ ]:
# Perform complete integration
integrated_df = integrator.integrate_all(
    iar_path='../data/raw/iar_transactions.csv',
    mbar_path='../data/raw/mbar_customers.csv',
    fraud_path='../data/raw/fraud_labels.csv',
    output_path='../data/processed/integrated_dataset.csv'
)

In [ ]:
# View integration statistics
print("Integration Statistics:")
for key, value in integrator.merge_stats.items():
    print(f"  {key}: {value}")

## 5. Exploratory Data Analysis

In [ ]:
# Fraud distribution
fraud_counts = integrated_df['is_fraud'].value_counts()
print(f"\nFraud Distribution:")
print(fraud_counts)
print(f"Fraud Rate: {integrated_df['is_fraud'].mean():.4%}")

# Visualize
plt.figure(figsize=(8, 5))
fraud_counts.plot(kind='bar')
plt.title('Fraud vs Legitimate Transactions')
plt.xlabel('Is Fraud')
plt.ylabel('Count')
plt.xticks([0, 1], ['Legitimate', 'Fraud'], rotation=0)
plt.show()

In [ ]:
# Missing value analysis
missing_summary = pd.DataFrame({
    'column': integrated_df.columns,
    'missing_count': integrated_df.isnull().sum().values,
    'missing_pct': (integrated_df.isnull().sum() / len(integrated_df) * 100).values
})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print("\nMissing Value Summary:")
display(missing_summary)

In [ ]:
# Transaction amount distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
integrated_df['amount'].hist(bins=50, edgecolor='black')
plt.title('Transaction Amount Distribution')
plt.xlabel('Amount')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
np.log1p(integrated_df['amount']).hist(bins=50, edgecolor='black')
plt.title('Log Transaction Amount Distribution')
plt.xlabel('Log(Amount + 1)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Fraud by channel
if 'channel' in integrated_df.columns:
    fraud_by_channel = integrated_df.groupby('channel')['is_fraud'].agg(['sum', 'mean', 'count'])
    fraud_by_channel.columns = ['fraud_count', 'fraud_rate', 'total_transactions']
    fraud_by_channel = fraud_by_channel.sort_values('fraud_rate', ascending=False)
    
    print("\nFraud by Channel:")
    display(fraud_by_channel)
    
    # Visualize
    fraud_by_channel['fraud_rate'].plot(kind='bar', figsize=(10, 5))
    plt.title('Fraud Rate by Channel')
    plt.xlabel('Channel')
    plt.ylabel('Fraud Rate')
    plt.xticks(rotation=45)
    plt.show()

## 6. Save Integrated Dataset

In [ ]:
# Dataset is already saved by integrator
print(f"Integrated dataset saved to: ../data/processed/integrated_dataset.csv")
print(f"Shape: {integrated_df.shape}")

## Next Steps

1. Proceed to notebook `02_data_cleaning.ipynb` for data cleaning
2. Then to `03_feature_engineering.ipynb` for feature creation
3. Finally to `04_feature_selection_modeling.ipynb` for feature selection and modeling